# Differential expression analysis


You have run the nf-core/rnaseq pipeline and checked the first quality control metrics of your fastq files. This was, however, only the primary analysis and we want to take it further.

Due to the computational demand of the pipeline, you only ran the pipeline on two of the 16 samples in the study yesterday. We provide you an essential output of nf-core/rnaseq pipeline in the `data` folder: It contains the combined epression matrix as produced by Salmon, which provides transcript levels for each gene (rows) and each sample (columns).


We would now like to understand exactly the difference between the expression in our groups of mice. 
Which pipeline would you use for this?

I would use nf-core/differentialabundance for the secondary analysis. It takes the Salmon expression matrix for all 16 samples, together with sample metadata, and tests which genes differ between groups.
I would first compare SNI vs Sham within the saline group, then Oxy vs Sal within SNI and Oxy vs Sal within Sham. Using all 16 samples gives four mice per condition combination, rather than relying on the two samples used to test the RNA-seq workflow.

Have a close look at the pipeline's "Usage" page on the [nf-core docs](nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

In [ ]:
import pandas as pd

metadata = pd.read_csv("data/samplesheet.csv")

number = metadata.groupby(["injury", "condition"]).cumcount() + 1
condition_name = metadata["condition"].replace({"sal": "Sal", "oxy": "oxy"})

metadata["sample"] = (
    metadata["injury"] + "_" + condition_name + "_" + number.astype(str)
)

samplesheet = metadata[["sample", "condition", "injury", "group"]]

matrix_samples = pd.read_csv(
    "data/salmon.merged.gene_counts.tsv", sep="\t", nrows=0
).columns.tolist()[2:]

assert set(samplesheet["sample"]) == set(matrix_samples)
samplesheet.to_csv("data/samplesheet.csv", index=False)
samplesheet

,sample,condition,injury,group
0,Sham_Sal_1,sal,Sham,Sham_sal
1,Sham_oxy_1,oxy,Sham,Sham_oxy
2,SNI_Sal_1,sal,SNI,SNI_sal
3,SNI_oxy_1,oxy,SNI,SNI_oxy
4,Sham_Sal_2,sal,Sham,Sham_sal
5,Sham_oxy_2,oxy,Sham,Sham_oxy
6,SNI_Sal_2,sal,SNI,SNI_sal
7,SNI_oxy_2,oxy,SNI,SNI_oxy
8,Sham_Sal_3,sal,Sham,Sham_sal
9,Sham_oxy_3,oxy,Sham,Sham_oxy


I created a samplesheet using the sample IDs in the Salmon gene-count matrix. I matched each SRX sample ID to its SRR run ID, then added the condition (sal or oxy) and injury group (Sham or SNI) from the provided Excel file. There are 16 samples in total, with four samples in each of the four groups. I saved the table as day_03/samplesheet.csv.

I created four contrasts to compare SNI with Sham under each treatment, and oxycodone with saline within each injury group. The contrast file is saved as day_03/contrasts.csv. Each contrast uses the group column in my samplesheet.

Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [ ]:
nextflow run nf-core/differentialabundance \
  -profile rnaseq,docker \
  -c desktop.config \
  --input data/samplesheet.csv \
  --contrasts data/contrasts.csv \
  --matrix data/salmon.merged.gene_counts.tsv \
  --features_id_col gene_id \
  --features_name_col gene_name \
  --features_metadata_cols gene_id,gene_name \
  --outdir differentialabundance_results \
  -resume

Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

- -profile rnaseq,docker selects the RNA-seq analysis settings with DESeq2 and runs the required software in Docker containers.
- --input data/samplesheet.csv provides sample names and their treatment and injury groups. The sample names match the matrix column names.
- --contrasts data/contrasts.csv defines which groups to compare and which group is the reference.
- --matrix data/salmon.merged.gene_counts.tsv supplies the Salmon gene counts, with genes as rows and samples as columns.
- --features_id_col gene_id identifies genes using their unique IDs.
- --features_name_col gene_name uses gene names in plots and reports.
- --features_metadata_cols gene_id,gene_name selects the annotation columns available in the matrix, allowing analysis without a separate GTF.
- --outdir differentialabundance_results specifies where to save the results.


What were the outputs of the pipeline?

In [2]:
#!TODO

Would you exclude any samples? If yes, which and why?

How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state?

Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.